# Install fenicsx

In [ ]:
# One-cell FEniCSx install + minimal vector space test for fresh Google Colab
# Do not pip-install ufl/ffcx separately.

import sys
import subprocess
import importlib
import textwrap
import time
from pathlib import Path

INSTALL_URL = "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"
INSTALL_SH = "/tmp/fenicsx-install.sh"
INSTALL_LOG = "/tmp/fenicsx-install.log"

print("Python:", sys.version)
print("Executable:", sys.executable)

def run(cmd, check=True):
    print(f"\n$ {cmd}")
    p = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return p

# Fresh-runtime sanity check
try:
    import dolfinx
    raise RuntimeError(
        "dolfinx is already present. This runtime is not clean. "
        "Use Runtime -> Disconnect and delete runtime, then rerun this cell."
    )
except ModuleNotFoundError:
    pass

# Download installer
run(f'wget -q "{INSTALL_URL}" -O "{INSTALL_SH}"')
run(f"test -s {INSTALL_SH}")

# Run installer with short output and one retry for GitHub 429/rate-limit failures
success = False
for attempt in range(1, 3):
    print(f"\nFEniCSx install attempt {attempt}/2")
    p = subprocess.run(
        f"bash {INSTALL_SH} > {INSTALL_LOG} 2>&1",
        shell=True,
        text=True,
    )

    log_tail = Path(INSTALL_LOG).read_text(errors="replace")[-4000:]
    print("\n--- installer log tail ---")
    print(log_tail)
    print("--- end installer log tail ---")

    if p.returncode == 0:
        success = True
        break

    if "429 Too Many Requests" in log_tail and attempt == 1:
        print("\nGitHub rate limit hit. Retrying once after a short pause.")
        time.sleep(30)
    else:
        raise RuntimeError(
            "FEM-on-Colab installer failed. "
            "See /tmp/fenicsx-install.log in the Colab runtime for the full log."
        )

if not success:
    raise RuntimeError("FEM-on-Colab installer failed after retry.")

importlib.invalidate_caches()

# Imports
import dolfinx
import ufl
import basix
import ffcx
from mpi4py import MPI
from dolfinx import mesh, fem

print("\nVersions")
print("dolfinx:", dolfinx.__version__)
print("ufl:", ufl.__version__)
print("basix:", basix.__version__)
print("ffcx:", ffcx.__version__)

# ---------------------------------------------------------------------
# Minimal FEniCSx scalar space test
# ---------------------------------------------------------------------
domain = mesh.create_unit_square(MPI.COMM_WORLD, 4, 4)
Q = fem.functionspace(domain, ("Lagrange", 1))
q = fem.Function(Q)
q.interpolate(lambda x: x[0] + x[1])

print("\nScalar function space test OK")
print("Q dofs:", Q.dofmap.index_map.size_local * Q.dofmap.index_map_bs)

# ---------------------------------------------------------------------
# Minimal FEniCSx vector space test
# ---------------------------------------------------------------------
V = fem.functionspace(domain, ("Lagrange", 1, (domain.geometry.dim,)))
u = fem.Function(V)
u.interpolate(lambda x: (x[0], x[1]))

print("\nVector function space test OK")
print("V block size:", V.dofmap.index_map_bs)
print("V local scalar dofs:", V.dofmap.index_map.size_local * V.dofmap.index_map_bs)

print("\nFEniCSx / dolfinx install and vector space test OK")

# Install pyvista

In [ ]:
try:
    import pyvista
    import pyvirtualdisplay
    import vtk

    if not hasattr(pyvista, "start_xvfb"):
        raise AttributeError("pyvista.start_xvfb is missing")

    print("PyVista setup looks OK")
    print("PyVista version:", pyvista.__version__)

except Exception as e:
    print("Installing/upgrading PyVista/VTK/Xvfb because:", e)

    !apt-get update -qq
    !apt-get install -qq xvfb libgl1-mesa-glx
    !pip install -q -U "pyvista[jupyter]" pyvirtualdisplay vtk

    print("Install done. Restart runtime now: Runtime → Restart runtime")

# Define pyvista plotter

In [ ]:
import numpy as np
import pyvista

import os
from pyvirtualdisplay import Display
import pyvista

_pv_display = None

def ensure_pyvista_colab():
    global _pv_display

    os.environ["PYVISTA_OFF_SCREEN"] = "true"

    try:
        pyvista.set_jupyter_backend("static")
    except Exception as e:
        print("Could not set PyVista backend:", e)

    try:
        pyvista.global_theme.notebook = True
    except Exception:
        pass

    if hasattr(pyvista, "start_xvfb"):
        try:
            pyvista.start_xvfb()
            return
        except Exception as e:
            print("pyvista.start_xvfb failed, using pyvirtualdisplay instead:", e)

    if _pv_display is None:
        _pv_display = Display(visible=False, size=(1024, 768))
        _pv_display.start()

def plot_pv(domain, uh, disph=None, clear=0, factor_warp=1.0, do_3d=True):
    points = domain.geometry.x
    cells = domain.geometry.dofmap
    n_cells = cells.shape[0]

    cells2 = np.hstack((np.full((n_cells, 1), 3, dtype=cells.dtype), cells))
    celltypes = np.full(n_cells, pyvista.CellType.TRIANGLE, dtype=np.uint8)
    grid = pyvista.UnstructuredGrid(cells2, celltypes, points)

    # Scalars: temperature
    grid.point_data["uh"] = uh.x.array

    # Vectors: displacement (optional)
    if disph is None:
        disp3 = np.zeros((len(points), 3))
    else:
        disp3 = np.hstack((disph.x.array.reshape(-1, 2), np.zeros((len(points), 1))))
    grid.point_data["disph"] = disp3

    #pyvista.set_jupyter_backend("static")
    #pyvista.global_theme.notebook = True
    #pyvista.start_xvfb()
    ensure_pyvista_colab()

    # ---- 2D view (warped if disph exists)
    plotter = pyvista.Plotter(off_screen=True)
    base2d = grid.warp_by_vector(vectors="disph", factor=factor_warp) if disph is not None else grid

    plotter.add_mesh(base2d, scalars="uh", cmap="viridis", show_edges=True)
    plotter.camera_position = [
        (0, 0, 3),
        (0, 0, 0),
        (0, 1, 0),
    ]

    if clear == 1:
        from google.colab import output
        output.clear()

    plotter.show()

    if not do_3d:
        return

    # ---- 3D build: rotate/extrude (warped if available, otherwise unwarped)
    base3d = base2d  # already warped or not
    surf = base3d.extract_surface()
    surf = surf.rotate_x(90)

    # Your original mirror/rotation setup
    surf_miror = surf.rotate_z(240)
    solid = surf.extrude_rotate(resolution=100, angle=240.0)

    pl = pyvista.Plotter(off_screen=True)
    _ = pl.add_mesh(solid, show_edges=False, reset_camera=True)
    _ = pl.add_mesh(surf, show_edges=True, reset_camera=True)
    _ = pl.add_mesh(surf_miror, show_edges=True, reset_camera=True)

    pl.camera_position = [
        (1.0, -3.0, 1.0),
        (0.0, 0.0, 0.0),
        (0.0, 0.0, 1.0),
    ]
    pl.show()

# fenicx code

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import mesh, fem, io


# ---------------------------------------------------------------------
# Small helpers
# ---------------------------------------------------------------------

def grad_cyl(v):
    """Gradient in the (r, z) computational plane."""
    return ufl.as_vector([ufl.Dx(v, 0), ufl.Dx(v, 1)])


eps0 = 1e-10


def eps_cyl(u, rfun):
    """Axisymmetric strain tensor in (r,z), embedded in 3D."""
    ur, uz = u[0], u[1]
    r_ = rfun + eps0

    return ufl.sym(
        ufl.as_tensor(
            [
                [ufl.Dx(ur, 0), 0.0,        ufl.Dx(ur, 1)],
                [0.0,           ur / r_,    0.0],
                [ufl.Dx(uz, 0), 0.0,        ufl.Dx(uz, 1)],
            ]
        )
    )


def sigma_iso(eps, lmbda, mu):
    return lmbda * ufl.tr(eps) * ufl.Identity(3) + 2.0 * mu * eps


def eps_therm(theta, alpha):
    return alpha * theta * ufl.Identity(3)


def subspace_dofs_geometrical(V_sub, V_parent, marker):
    dofs = fem.locate_dofs_geometrical((V_sub, V_parent), marker)

    if isinstance(dofs, (tuple, list)):
        out = dofs[0]
    else:
        dofs = np.asarray(dofs)
        out = dofs[:, 0] if (dofs.ndim == 2 and dofs.shape[1] == 2) else dofs

    return np.asarray(out, dtype=np.int32)


def _get(p: Dict[str, Any], key: str, default):
    return p[key] if key in p else default


def _to_step_set(steps) -> set[int]:
    """steps: array-like 1-based indices; [] or None => empty set."""
    if steps is None:
        return set()

    arr = np.asarray(steps, dtype=int).ravel()
    return set(int(s) for s in arr.tolist())


def _safe_remove_xdmf_pair(path: Path):
    """Remove foo.xdmf and foo.h5 if they exist."""
    h5 = path.with_suffix(".h5")

    for p in (path, h5):
        try:
            if p.exists():
                p.unlink()
        except Exception:
            pass


# ---------------------------------------------------------------------
# Source term
# ---------------------------------------------------------------------

class TimeGatedGaussian:
    def __init__(self):
        self.t = 0.0
        self.power = 0.0
        self.wr = 1.0
        self.wz = 1.0
        self.heating_duration = 0.0
        self.r0 = 0.0
        self.z0 = 0.0

    def update(
        self,
        *,
        power: float,
        wr: float,
        wz: float,
        heating_duration: float,
        r0: float,
        z0: float,
    ):
        self.power = float(power)
        self.wr = float(wr)
        self.wz = float(wz)
        self.heating_duration = float(heating_duration)
        self.r0 = float(r0)
        self.z0 = float(z0)

    def __call__(self, x):
        rr = ((x[0] - self.r0) / self.wr) ** 2
        zz = ((x[1] - self.z0) / self.wz) ** 2
        g = self.power * np.exp(-0.5 * (rr + zz))
        return g * (self.t <= self.heating_duration)


# ---------------------------------------------------------------------
# Main problem class
# ---------------------------------------------------------------------

class AxisymHeatProblem:
    def __init__(self, comm=MPI.COMM_WORLD):
        self.comm = comm

        # Thermal mesh/state
        self._geom_sig: Optional[Tuple] = None
        self.domain = None
        self.V = None
        self.bcs = None
        self.r_weight = None

        self.uh = None
        self.u_n = None
        self.f = None
        self.source = TimeGatedGaussian()

        self.v = None
        self.du = None

        # Mechanics state, built on demand
        self._meca_built = False
        self.V_vec = None
        self.disph = None
        self._T_meca = None
        self._a_meca = None
        self._L_meca = None
        self._bc_meca = None
        self._A_meca = None
        self._ksp_meca = None
        self._b_meca = None

    # -----------------------------------------------------------------
    # Mesh and thermal space
    # -----------------------------------------------------------------

    def _build_mesh_if_needed(self, p: Dict[str, Any]):
        width = float(_get(p, "width", _get(p, "radius", 1.0)))
        thickness = float(_get(p, "thickness", 0.5))

        if "nx" in p and "ny" in p:
            nx = int(p["nx"])
            ny = int(p["ny"])
        else:
            nb = _get(p, "nb_elements", [60, 20])
            nx = int(nb[0])
            ny = int(nb[1])

        sig = (width, thickness, nx, ny)
        if self._geom_sig == sig and self.domain is not None:
            return

        self._geom_sig = sig

        self.domain = mesh.create_rectangle(
            self.comm,
            [np.array([0.0, 0.0]), np.array([width, thickness])],
            [nx, ny],
            cell_type=mesh.CellType.triangle,
        )

        self.V = fem.functionspace(self.domain, ("CG", 1))

        def boundary_bottom(x):
            return np.isclose(x[1], 0.0)

        dofs = fem.locate_dofs_geometrical(self.V, boundary_bottom)
        self.bcs = [fem.dirichletbc(PETSc.ScalarType(0.0), dofs, self.V)]

        self.r_weight = fem.Function(self.V)
        self.r_weight.interpolate(lambda x: np.maximum(np.abs(x[0]), 1e-14))

        self.uh = fem.Function(self.V)
        self.uh.name = "uh"

        self.u_n = fem.Function(self.V)
        self.u_n.name = "u_n"
        self.u_n.x.array[:] = 0.0

        self.f = fem.Function(self.V)

        self.v = ufl.TestFunction(self.V)
        self.du = ufl.TrialFunction(self.V)

        # Geometry changed, so mechanics must be rebuilt
        self._meca_built = False
        self.disph = None

    # -----------------------------------------------------------------
    # Mechanics
    # -----------------------------------------------------------------

    def _build_mechanics(self, meca_params: dict):
        assert self.domain is not None
        assert self.V is not None
        assert self.r_weight is not None

        domain = self.domain
        r = self.r_weight

        self.V_vec = fem.functionspace(domain, ("CG", 1, (2,)))

        u = ufl.TrialFunction(self.V_vec)
        v = ufl.TestFunction(self.V_vec)

        self._T_meca = fem.Function(self.V)
        self._T_meca.name = "T_meca"

        E = float(meca_params.get("E", 1.0))
        nu = float(meca_params.get("nu", 0.3))
        alpha = float(meca_params.get("alpha", 1.0))

        mu = E / (2.0 * (1.0 + nu))
        lmbda = E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))

        eps_u = eps_cyl(u, r)
        eps_v = eps_cyl(v, r)
        eps_th = eps_therm(self._T_meca, alpha)

        a_ufl = ufl.inner(sigma_iso(eps_u, lmbda, mu), eps_v) * r * ufl.dx
        L_ufl = ufl.inner(sigma_iso(eps_th, lmbda, mu), eps_v) * r * ufl.dx

        self._a_meca = fem.form(a_ufl)
        self._L_meca = fem.form(L_ufl)

        def boundary_bottom(x):
            return np.isclose(x[1], 0.0)

        def boundary_left(x):
            return np.isclose(x[0], 0.0)

        def corner_bottom_left(x):
            return np.isclose(x[0], 0.0) & np.isclose(x[1], 0.0)

        facet_dim = domain.topology.dim - 1

        bottom_dofs_uz = subspace_dofs_geometrical(
            self.V_vec.sub(1),
            self.V_vec,
            boundary_bottom,
        )
        bc_bottom_uz = fem.dirichletbc(
            PETSc.ScalarType(0.0),
            bottom_dofs_uz,
            self.V_vec.sub(1),
        )

        left_facets = mesh.locate_entities_boundary(domain, facet_dim, boundary_left)
        left_dofs_ur = fem.locate_dofs_topological(
            self.V_vec.sub(0),
            facet_dim,
            left_facets,
        )
        left_dofs_ur = np.asarray(left_dofs_ur, dtype=np.int32)
        bc_left_ur = fem.dirichletbc(
            PETSc.ScalarType(0.0),
            left_dofs_ur,
            self.V_vec.sub(0),
        )

        corner_dofs = fem.locate_dofs_geometrical(self.V_vec, corner_bottom_left)
        corner_dofs = np.asarray(corner_dofs, dtype=np.int32)
        u_pin = np.array([0.0, 0.0], dtype=PETSc.ScalarType)
        bc_pin = fem.dirichletbc(u_pin, corner_dofs, self.V_vec)

        self._bc_meca = [bc_bottom_uz, bc_left_ur, bc_pin]

        self._A_meca = fem.petsc.assemble_matrix(self._a_meca, bcs=self._bc_meca)
        self._A_meca.assemble()

        self._ksp_meca = PETSc.KSP().create(domain.comm)
        self._ksp_meca.setOperators(self._A_meca)
        self._ksp_meca.setType(PETSc.KSP.Type.PREONLY)
        self._ksp_meca.getPC().setType(PETSc.PC.Type.LU)
        self._ksp_meca.setFromOptions()

        self.disph = fem.Function(self.V_vec)
        self.disph.name = "disph"
        self.disph.x.array[:] = 0.0

        self._b_meca = self.disph.x.petsc_vec.duplicate()
        self._b_meca.set(0.0)

        self._meca_built = True

    def solve_mechanics(
        self,
        meca_params: dict,
        uh: Optional[fem.Function] = None,
        *,
        rebuild: bool = False,
    ) -> fem.Function:
        if uh is None:
            assert self.uh is not None
            uh = self.uh

        if rebuild or (not self._meca_built):
            self._build_mechanics(meca_params)

        assert self.disph is not None
        assert self._T_meca is not None
        assert self._b_meca is not None
        assert self._L_meca is not None
        assert self._a_meca is not None
        assert self._bc_meca is not None
        assert self._ksp_meca is not None

        self._T_meca.x.array[:] = uh.x.array

        self._b_meca.set(0.0)
        fem.petsc.assemble_vector(self._b_meca, self._L_meca)
        fem.petsc.apply_lifting(self._b_meca, [self._a_meca], [self._bc_meca])
        self._b_meca.ghostUpdate(
            addv=PETSc.InsertMode.ADD_VALUES,
            mode=PETSc.ScatterMode.REVERSE,
        )
        fem.petsc.set_bc(self._b_meca, self._bc_meca)

        self._ksp_meca.solve(self._b_meca, self.disph.x.petsc_vec)
        self.disph.x.scatter_forward()

        return self.disph

    # -----------------------------------------------------------------
    # Thermal solve, optional mechanics, optional sensors/output
    # -----------------------------------------------------------------

    def solve(
        self,
        parameters: Dict[str, Any],
        *,
        meca_params: Optional[dict] = None,
        thermal_output_steps=None,
        meca_output_steps=None,
        same_file: bool = False,
        out_dir: str = "out",
        thermal_name: str = "thermal.xdmf",
        meca_name: str = "meca.xdmf",
        both_name: str = "thermo_meca.xdmf",
        reset_initial_condition: bool = True,
        sensors_xy: Optional[np.ndarray] = None,
        sensor_output_steps=None,
        verbose: bool = True,
    ) -> Tuple[List[int], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:

        self._build_mesh_if_needed(parameters)

        assert self.domain is not None
        assert self.V is not None
        assert self.bcs is not None
        assert self.r_weight is not None
        assert self.uh is not None
        assert self.u_n is not None
        assert self.f is not None
        assert self.v is not None
        assert self.du is not None

        # Time
        t = float(_get(parameters, "T_init", 0.0))
        T_end = float(_get(parameters, "T_end", 0.01))
        n_steps = int(_get(parameters, "n_time_steps", 100))
        dt = (T_end - t) / n_steps

        # Source
        power = float(_get(parameters, "power", 1.0))
        wr = float(_get(parameters, "width_gaussian", 0.25))
        wz = max(1e-15, float(_get(parameters, "ratio_width_z", 0.1)) * wr)

        thickness = float(_get(parameters, "thickness", self._geom_sig[1]))
        heating_duration = float(_get(parameters, "heating_duration", 0.5 * T_end))
        z0 = float(_get(parameters, "source_z0", thickness))
        r0 = float(_get(parameters, "source_r0", 0.0))

        self.source.update(
            power=power,
            wr=wr,
            wz=wz,
            heating_duration=heating_duration,
            r0=r0,
            z0=z0,
        )

        # Material
        thermal_capacity = float(_get(parameters, "thermal_capacity", 1.0))
        diffusion_coeff = float(_get(parameters, "diffusion_coeff", 1.0))
        advection_coeff = float(_get(parameters, "advection_coeff", 1.0))

        # Initial condition
        if reset_initial_condition:
            self.u_n.x.array[:] = 0.0
            self.uh.x.array[:] = 0.0

            if self.disph is not None:
                self.disph.x.array[:] = 0.0

        # Linear thermal problem
        uh = self.uh
        u_n = self.u_n
        du = self.du
        v = self.v
        r = self.r_weight

        a_ufl = (
            thermal_capacity * (1.0 / dt) * du * v * r * ufl.dx
            + diffusion_coeff * ufl.dot(grad_cyl(du), grad_cyl(v)) * r * ufl.dx
            + advection_coeff * du * v * r * ufl.ds
        )

        L_ufl = (
            thermal_capacity * (1.0 / dt) * u_n * v * r * ufl.dx
            + self.f * v * r * ufl.dx
        )

        a = fem.form(a_ufl)
        L = fem.form(L_ufl)

        A = fem.petsc.assemble_matrix(a, bcs=self.bcs)
        A.assemble()

        b = uh.x.petsc_vec.duplicate()

        ksp = PETSc.KSP().create(self.domain.comm)
        ksp.setOperators(A)
        ksp.setType(_get(parameters, "ksp_type", "preonly"))
        ksp.getPC().setType(_get(parameters, "pc_type", "lu"))
        ksp.setFromOptions()

        # Output selectors
        thermal_set = _to_step_set(thermal_output_steps)
        meca_set = _to_step_set(meca_output_steps)

        want_thermal = len(thermal_set) > 0
        want_meca = (meca_params is not None) and (len(meca_set) > 0)

        if same_file:
            write_union_set = thermal_set | meca_set
            want_both_out = len(write_union_set) > 0
            want_thermal = False
            want_meca = False
        else:
            write_union_set = set()
            want_both_out = False

        # Sensors
        times = None
        temps = None
        ok = None

        record_set = _to_step_set(sensor_output_steps)
        if sensor_output_steps is None:
            record_set = set(range(1, n_steps + 1))

        if sensors_xy is not None:
            from dolfinx import geometry

            sensors_xy = np.asarray(sensors_xy, dtype=np.float64)
            if sensors_xy.ndim != 2 or sensors_xy.shape[1] != 2:
                raise ValueError("sensors_xy must be an (N,2) array of (r,z) points.")

            n_sensors = sensors_xy.shape[0]
            points = np.zeros((n_sensors, 3), dtype=np.float64)
            points[:, : self.domain.geometry.dim] = sensors_xy[:, : self.domain.geometry.dim]

            bb = geometry.bb_tree(self.domain, self.domain.topology.dim)
            cand = geometry.compute_collisions_points(bb, points)
            coll = geometry.compute_colliding_cells(self.domain, cand, points)

            cells = np.full(n_sensors, -1, dtype=np.int32)
            for i in range(n_sensors):
                cands = coll.links(i)
                if len(cands) > 0:
                    cells[i] = int(cands[0])

            ok = cells >= 0
            idx_ok = np.where(ok)[0]

            times_list: List[float] = []
            temps_list: List[np.ndarray] = []

        # Output files
        out_path = Path(out_dir)
        out_path.mkdir(parents=True, exist_ok=True)

        xdmfT = None
        xdmfM = None
        xdmfB = None

        if want_thermal:
            p = out_path / thermal_name
            _safe_remove_xdmf_pair(p)
            xdmfT = io.XDMFFile(self.domain.comm, str(p), "w")
            xdmfT.write_mesh(self.domain)

        if want_meca:
            p = out_path / meca_name
            _safe_remove_xdmf_pair(p)
            xdmfM = io.XDMFFile(self.domain.comm, str(p), "w")
            xdmfM.write_mesh(self.domain)

        if want_both_out:
            p = out_path / both_name
            _safe_remove_xdmf_pair(p)
            xdmfB = io.XDMFFile(self.domain.comm, str(p), "w")
            xdmfB.write_mesh(self.domain)

        its_history: List[int] = []

        try:
            meca_built_in_run = False

            for step in range(1, n_steps + 1):
                t += dt

                # Update source
                self.source.t = t
                self.f.interpolate(self.source)

                # Thermal linear solve
                b.set(0.0)
                fem.petsc.assemble_vector(b, L)
                fem.petsc.apply_lifting(b, [a], [self.bcs])
                b.ghostUpdate(
                    addv=PETSc.InsertMode.ADD_VALUES,
                    mode=PETSc.ScatterMode.REVERSE,
                )
                fem.petsc.set_bc(b, self.bcs)

                ksp.solve(b, uh.x.petsc_vec)
                uh.x.scatter_forward()

                its = ksp.getIterationNumber()
                its_history.append(int(its))
                converged = ksp.getConvergedReason() > 0

                if verbose and self.domain.comm.rank == 0:
                    print(
                        f"step {step}/{n_steps}, "
                        f"t={t:.5e}, "
                        f"its={its}, "
                        f"converged={converged}"
                    )

                # Advance temperature
                u_n.x.array[:] = uh.x.array

                # Thermal output
                if xdmfT is not None and (step in thermal_set):
                    xdmfT.write_function(uh, t)

                # Mechanics output
                if xdmfM is not None and (step in meca_set):
                    self.solve_mechanics(
                        meca_params,
                        uh=uh,
                        rebuild=not meca_built_in_run,
                    )
                    meca_built_in_run = True
                    xdmfM.write_function(self.disph, t)

                # Combined output
                if xdmfB is not None and (step in write_union_set):
                    if self.disph is None:
                        if meca_params is None:
                            raise RuntimeError("same_file=True requires meca_params.")
                        self._build_mechanics(meca_params)
                        meca_built_in_run = True

                    if (meca_params is not None) and (step in meca_set):
                        self.solve_mechanics(
                            meca_params,
                            uh=uh,
                            rebuild=not meca_built_in_run,
                        )
                        meca_built_in_run = True

                    xdmfB.write_function(uh, t)
                    xdmfB.write_function(self.disph, t)

                # Sensors
                if sensors_xy is not None and (step in record_set):
                    vals = np.full((points.shape[0],), np.nan, dtype=np.float64)

                    if idx_ok.size > 0:
                        v_ok = uh.eval(points[idx_ok], cells[idx_ok])
                        vals[idx_ok] = np.asarray(v_ok, dtype=np.float64).reshape(-1)

                    times_list.append(t)
                    temps_list.append(vals)

            if sensors_xy is not None:
                times = np.asarray(times_list, dtype=np.float64)
                temps = np.asarray(temps_list, dtype=np.float64).T

            return its_history, times, temps, ok

        finally:
            if xdmfT is not None:
                xdmfT.close()
            if xdmfM is not None:
                xdmfM.close()
            if xdmfB is not None:
                xdmfB.close()

# Run simulation

In [ ]:
parameters = {
    # problem:
    "T_init": 0.0,
    "T_end": 0.5,
    "n_time_steps": 100,
    "radius": 1.0,      # used as width if 'width' not provided
    "thickness": 0.5,
    "nx": 60,
    "ny": 20,
    "temp0": 393.0,

    # heat source
    "power": 1.0,
    "width_gaussian": 0.25,
    "ratio_width_z": 0.1,
    "heating_duration": 100,

    # material
    "thermal_capacity": 1.0,
    "advection_coeff": 1.0,
    "diffusion_coeff": 1.0,
    "radiation_coeff": 1.0e-13,
}
meca_params = {"E": 1., "nu": 0.3, "alpha": 1.0, "T_ref": 0.0}

thermal_output_steps = np.array([1, 25, 50, 100])
meca_output_steps = np.array([10, 20, 30, 40, 50, 100])

problem = AxisymHeatProblem()

its = problem.solve(
    parameters,
    meca_params=meca_params,
    thermal_output_steps=thermal_output_steps,
    meca_output_steps=meca_output_steps,
    same_file=True,              # two files, independent selectors
    out_dir="out"
)

In [ ]:
#plot_pv(problem.domain, problem.uh, None, clear=1)
plot_pv(problem.domain, problem.uh, problem.disph, clear=1, factor_warp=100)   # mechanics available

# Extract temperature history at points

In [ ]:
sensors = np.array([
    [0.1, 0.25],
    [0.5, 0.25],
    [0.9, 0.25],
])

its, times, temps, ok = problem.solve(
    parameters,
    sensors_xy=sensors,
    sensor_output_steps=None,          # record all time steps
    thermal_output_steps=[],           # no thermal file
    meca_output_steps=[],              # no meca file
    verbose = False
)

import matplotlib.pyplot as plt

print(times.shape)  # (n_rec,)

print(temps.shape)  # (N_sensors, n_t)
print(ok)           # which sensors were inside mesh

plt.plot(np.arange(temps.shape[1]),temps.T)
plt.show()

data = temps
parameters_truth = parameters.copy()